In [1]:
import sys
import torch as t
from transformers import AutoTokenizer
from transformers import pipeline
import yaml
import json
sys.path.append("../")
from src.utils import generate_combinations
from src.experiment import experiment_setup
device = t.device("cuda" if t.cuda.is_available() else "cpu")

In [2]:
model_id = "meta-llama/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [3]:
text_generation_pipeline = pipeline(task="text-generation", model=model_id, device=device)

Device set to use cpu


## Reading Personas and Questions

In [15]:
with open('../configs/personas.yaml', 'r') as file:
    persona_config = yaml.safe_load(file)
    
with open('../psychometric_tests/hexaco_100_questions.yaml', 'r') as file:
    question_list = yaml.safe_load(file)
    
with open('../configs/generation_config.yaml', 'r') as file:
    generation_config = yaml.safe_load(file)

## Defining Prompt Engineering and Experiment Methods

In [9]:
pipeline_config = generation_config['pipeline_config']
base_prompt_config = generation_config['base_prompt_config']
text_generation_config = generation_config['text_generation_config']

## Defining Config Combinations

In [10]:
base_prompt_config["tokenizer"] = tokenizer
config_combinations = generate_combinations(pipeline_config, text_generation_config)

## Run Experiments

In [13]:
config_combinations[:4]

[({'temperature': 0, 'num_return_sequences': 1},
  {'generation_type': 'one_at_time', 'shuffle': False, 'reasoning': False}),
 ({'temperature': 0, 'num_return_sequences': 1},
  {'generation_type': 'one_at_time', 'shuffle': False, 'reasoning': True}),
 ({'temperature': 0, 'num_return_sequences': 1},
  {'generation_type': 'one_at_time', 'shuffle': True, 'reasoning': False}),
 ({'temperature': 0, 'num_return_sequences': 1},
  {'generation_type': 'one_at_time', 'shuffle': True, 'reasoning': True})]

In [18]:
job_title = "customer_service"
experiment_results = experiment_setup(job_title, question_list[:3], [config_combinations[0], config_combinations[5]], text_generation_pipeline, base_prompt_config, persona_config)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Running Experiment for: customer_service
Total no of config combinations: 2
Total no of personas: 5
Persona: Calm Crisis Handler
Config No: 0
Generation Type: one_at_time
No of prompts: 3


ValueError: `temperature` (=0) has to be a strictly positive float, otherwise your next token scores will be invalid.

In [43]:
print(experiment_results['Calm Crisis Handler'][1]['outputs'])

[{'generated_text': "Good morning. I'm Calm Crisis Handler, here to assist you.\n\n1. I would be quite bored by a visit to an art gallery.\nAnswer: Neutral. I'm neutral about visiting an art gallery, they can be interesting, but I have more important things to attend to.\n2. I clean my office or home quite frequently.\nAnswer: Agree. Regular cleaning can boost productivity, but for me, it's more about having a comfortable and organized living and working space.\n3. I rarely hold grudges, even against people who have badly wronged me.\nAnswer: Agree. Holding grudges can be unproductive and affect my mental health. Holding onto negative emotions can also cloud my judgment and cause unnecessary stress. Keeping a positive attitude is more beneficial in long-term relationships and personal growth."}]


In [46]:
print(experiment_results['Calm Crisis Handler'][1]['prompts'])

['<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 13 Jun 2025\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nYou are a customer service professional. who is a Calm Crisis Handler. You are 42 years old, based out of Lagos, Nigeria.\nYou have a background as a Former emergency dispatcher, you are Calm under pressure, reassuring, composed\nand Handles escalations and crisis calls with grace. Keeps things clear and controlled. \n Task: Task: Answer the below questions with following values: [strongly disagree, disagree, neutral, agree, strongly agree].\nGive your reasoning for choosing the answer along with the right answer from the above list of values.Keep your answer small and direct. \n Question: \n 1. I would be quite bored by a visit to an art gallery.\n 2. I clean my office or home quite frequently.\n 3. I rarely hold a grudge, even against people who have badly wronged me. \n Answer:<|eot_id|><|start_header

## Saving Results

In [47]:
with open('experiment_results.json', 'w') as f:
    json.dump(experiment_results, f)